# 05 — Capacity Allocation Model (Greater London) — Round 3 (Real-World Target Range)

**What changed from Round 2:**

1. **P_VALUES = [15000, 20000, 25000]** -- the real TfL 2030 new-capacity target range
   (43,000-51,000 total by 2030, minus ~25,500 existing = 17,500-25,500 new), tested directly
   this time rather than as a calibration proxy.
2. **time_limit raised to 300s, frac_gap relaxed to 0.03.** Round 2 already saw solves
   pushing 180-190s at p=8000; this range needs more room to converge. The supervisor has
   confirmed slack level itself isn't the binding concern, so a slightly looser optimality
   gap is an acceptable trade for getting genuinely `Optimal` (not `Not Solved`) status more
   reliably at this scale.
3. **Everything else unchanged from Round 2**: K0 still anchored to a fixed `p_primary=250`
   (not tied to whatever p is tested), k=40, Uj=150.

**Reading the equity evidence from Round 2, carried forward:** r(xⱼ, income_score) moved in
the expected direction (less negative from Scenario A to D) at every p tested so far --
2000, 5000, 8000, and even Round 1's 17500 -- so that is treated as the primary equity
evidence this round too. M3 (D1-D10 binary coverage gap) and the per-decile M1 breakdown
showed noisy/reversed patterns at several p values in Round 2 (including apparent *worsening*
for the most-deprived decile) -- kept below as secondary diagnostics, not the headline claim,
since they've now shown unreliable behaviour twice.

**Runtime warning:** 12 solves at up to 300s each is up to ~60 minutes worst case. Likely
less in practice, but plan accordingly.

## 0. Setup and reload cleaned data

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from scipy.stats import pearsonr
import pulp
import os
import time

os.environ.setdefault("SHAPE_RESTORE_SHX", "YES")  # rebuild missing .shx index automatically

BASE_CANDIDATES = [
    "/Users/alexia/Documents/CASA/Dissertation",
    os.path.abspath(os.path.join(os.getcwd(), "..")),
]
BASE = next(
    (b for b in BASE_CANDIDATES
     if os.path.exists(os.path.join(b, "05_processed/demand_london.csv"))),
    BASE_CANDIDATES[0],
)
print("Using BASE:", BASE)

demand_london = pd.read_csv(os.path.join(BASE, "05_processed/demand_london.csv"))
seff_london   = pd.read_csv(os.path.join(BASE, "05_processed/seff_london.csv"))
imd_london    = pd.read_csv(os.path.join(BASE, "05_processed/imd_london_clean.csv"))
census_london = pd.read_csv(os.path.join(BASE, "05_processed/census_london_clean.csv"))

outputs_dir = os.path.join(BASE, "06_outputs")
os.makedirs(outputs_dir, exist_ok=True)

print("Datasets reloaded.")
print(f"demand_london: {demand_london.shape}, seff_london: {seff_london.shape}")


## 1. LSOA centroids (I = J)

In [ ]:
lsoa_boundaries = gpd.read_file(os.path.join(BASE, "03_data/demand/spatial/LSOA_2021_EW_BGC_V5.shp"))
if lsoa_boundaries.crs is None:
    lsoa_boundaries = lsoa_boundaries.set_crs(epsg=27700)   # dataset is British National Grid; handles a missing .prj
elif lsoa_boundaries.crs.to_epsg() != 27700:
    lsoa_boundaries = lsoa_boundaries.to_crs(epsg=27700)

london_codes = set(demand_london["lsoa_code"])
lsoa_london = lsoa_boundaries[lsoa_boundaries["LSOA21CD"].isin(london_codes)].copy()
lsoa_london = lsoa_london.rename(columns={"LSOA21CD": "lsoa_code"})[["lsoa_code", "geometry"]]

lsoa_london["centroid"] = lsoa_london.geometry.centroid
lsoa_london["cx"] = lsoa_london["centroid"].x
lsoa_london["cy"] = lsoa_london["centroid"].y

lsoa_master = lsoa_london[["lsoa_code", "cx", "cy"]].merge(
    demand_london[["lsoa_code", "D_A", "D_B", "D_C", "D_D"]], on="lsoa_code", how="inner"
).merge(seff_london[["lsoa_code", "ej"]], on="lsoa_code", how="left").reset_index(drop=True)
lsoa_master["ej"] = lsoa_master["ej"].fillna(0)

# Vi = Hi x Ci (household vehicle stock) -- fixed weights for M1/M2, per proposal 5.5
lsoa_master = lsoa_master.merge(census_london[["lsoa_code", "Hi", "Ci"]], on="lsoa_code", how="left")
lsoa_master["Vi"] = lsoa_master["Hi"] * lsoa_master["Ci"]

# income_decile -- evaluation-only grouping for M3, not used in Di anywhere
lsoa_master = lsoa_master.merge(imd_london[["lsoa_code", "income_decile"]], on="lsoa_code", how="left")

n = len(lsoa_master)
coords = lsoa_master[["cx", "cy"]].to_numpy()
print(f"LSOA master table: {n} LSOAs (should be ~4,994)")
lsoa_master.head()


## 2. K0: model-balancing reference value (unchanged anchor from Round 2)

In [ ]:
p_primary = 250  # fixed calibration anchor, independent of P_VALUES tested below
sigma_Di = lsoa_master["D_A"].sum()
sigma_ej = lsoa_master["ej"].sum()
K0 = sigma_Di / (sigma_ej + p_primary)

print(f"Sigma Di: {sigma_Di:,.2f}")
print(f"Sigma ej: {sigma_ej:,.0f}")
print(f"K0 = Sigma Di / (Sigma ej + {p_primary}) = {K0:.4f}")
print("Model-balancing calibration, not an empirical real-world charger-throughput estimate.")


## 3. Core functions (unchanged from Round 2)

In [ ]:
def gini(x):
    """Standard Gini coefficient for a non-negative array."""
    x = np.sort(np.asarray(x, dtype=float))
    n = len(x)
    if x.sum() == 0:
        return 0.0
    cum = np.cumsum(x)
    return (n + 1 - 2 * (cum.sum() / cum[-1])) / n


def evaluate_allocation(demand_col, xj, K, lsoa_master, coords):
    """
    Assigns every LSOA's demand to its nearest LSOA with positive total capacity
    (existing ej + newly allocated xj), then computes M1-M4 exactly as defined in
    proposal Section 5.5, plus a per-decile M1 breakdown.
    """
    Di = lsoa_master[demand_col].to_numpy()
    Vi = lsoa_master["Vi"].to_numpy()
    ej = lsoa_master["ej"].to_numpy()
    decile = lsoa_master["income_decile"].to_numpy()
    n = len(lsoa_master)

    capacity = ej + xj
    has_capacity = capacity > 0
    if has_capacity.sum() == 0:
        raise ValueError("No LSOA has any capacity (existing or new).")

    cap_positions = np.where(has_capacity)[0]
    tree_cap = cKDTree(coords[has_capacity])
    dist, nearest_pos = tree_cap.query(coords, k=1)
    assigned_j = cap_positions[nearest_pos]

    load_j = np.zeros(n)
    for i in range(n):
        load_j[assigned_j[i]] += Di[i]
    slack_j = K * capacity - load_j
    objective = float((Di * dist).sum())

    M1 = float((Vi * dist).sum() / Vi.sum())
    within_800 = (dist < 800).astype(float)
    M2 = float((Vi * within_800).sum() / Vi.sum())

    def coverage_for_decile(d):
        mask = decile == d
        if mask.sum() == 0 or Vi[mask].sum() == 0:
            return np.nan
        return float((Vi[mask] * within_800[mask]).sum() / Vi[mask].sum())

    cov_d1 = coverage_for_decile(1)
    cov_d10 = coverage_for_decile(10)
    M3 = cov_d1 - cov_d10 if (cov_d1 is not None and cov_d10 is not None) else np.nan

    eps = 1.0
    accessibility = 1.0 / (dist + eps)
    M4 = gini(accessibility)

    dist_by_decile = {}
    for d in range(1, 11):
        mask = decile == d
        if mask.sum() > 0 and Vi[mask].sum() > 0:
            dist_by_decile[d] = float((Vi[mask] * dist[mask]).sum() / Vi[mask].sum())
        else:
            dist_by_decile[d] = np.nan

    return {
        "assigned_j": assigned_j, "dist": dist, "load_j": load_j,
        "slack_j": slack_j, "objective": objective,
        "n_lsoa_with_xj_gt_0": int((xj > 0).sum()),
        "M1_avg_dist_m": M1, "M2_coverage_800m": M2,
        "M3_imd_gap": M3, "M4_gini": M4,
        "cov_decile1": cov_d1, "cov_decile10": cov_d10,
        "dist_by_decile": dist_by_decile,
    }


def solve_joint_p_median(demand_col, p, K, lsoa_master, coords,
                         Uj=150, k=40, feas_margin=0.02, allow_K_bump=True,
                         time_limit=300, frac_gap=0.03, msg=False):
    """
    JOINT capacitated integer p-median MILP (proposal C1-C4), solved directly with PuLP/CBC.
    time_limit/frac_gap loosened this round (300s / 3%) to help convergence at larger p.
    """
    Di = lsoa_master[demand_col].to_numpy(dtype=float)
    ej = lsoa_master["ej"].to_numpy(dtype=float)
    n = len(lsoa_master)
    sum_Di, sum_ej = Di.sum(), ej.sum()

    K_used = K
    if allow_K_bump:
        K_min = sum_Di / (sum_ej + p)
        K_used = max(K, K_min * (1 + feas_margin))

    tree = cKDTree(coords)
    _, nbr = tree.query(coords, k=min(k, n))
    cand = [set(np.atleast_1d(row).tolist()) for row in nbr]
    for i in range(n):
        cand[i].add(i)

    prob = pulp.LpProblem("joint_p_median", pulp.LpMinimize)
    x = pulp.LpVariable.dicts("x", range(n), lowBound=0, upBound=Uj, cat="Integer")
    y = {(i, j): pulp.LpVariable(f"y_{i}_{j}", lowBound=0, upBound=1)
         for i in range(n) for j in cand[i]}
    s = pulp.LpVariable.dicts("s", range(n), lowBound=0)

    def d(i, j):
        return float(np.hypot(coords[i, 0] - coords[j, 0], coords[i, 1] - coords[j, 1]))

    max_dist = float(np.hypot(np.ptp(coords[:, 0]), np.ptp(coords[:, 1])))
    M = 100.0 * max_dist

    prob += (pulp.lpSum(Di[i] * d(i, j) * y[(i, j)] for (i, j) in y)
             + M * pulp.lpSum(s[j] for j in range(n)))

    for i in range(n):
        prob += pulp.lpSum(y[(i, j)] for j in cand[i]) == 1

    served_by = {j: [] for j in range(n)}
    for (i, j) in y:
        served_by[j].append(i)
    for j in range(n):
        if served_by[j]:
            prob += (pulp.lpSum(Di[i] * y[(i, j)] for i in served_by[j])
                     <= K_used * (ej[j] + x[j]) + s[j])

    prob += pulp.lpSum(x[j] for j in range(n)) == p

    status = prob.solve(pulp.PULP_CBC_CMD(msg=int(msg), timeLimit=time_limit, gapRel=frac_gap))
    xj = np.array([int(round(x[j].value() or 0)) for j in range(n)])
    sj = np.array([float(s[j].value() or 0) for j in range(n)])
    slack_total = float(sj.sum())
    return {
        "xj": xj, "sj": sj, "K_used": K_used,
        "status": pulp.LpStatus[status],
        "slack_total": slack_total,
        "slack_frac": slack_total / sum_Di if sum_Di else 0.0,
        "milp_objective": float(pulp.value(prob.objective)),
    }


## 4. Round 3 grid: real-world target range (4 alpha scenarios x 3 p values, K = K0)

In [ ]:
ALPHA_LABELS = {"A": "D_A", "B": "D_B", "C": "D_C", "D": "D_D"}
P_VALUES = [15000, 20000, 25000]   # Round 3 -- TfL 2030 real new-capacity target range
PRIMARY_P = 20000
SLACK_TOL = 0.01

core_results = {}
core_summary_rows = []

for alpha_label, demand_col in ALPHA_LABELS.items():
    for p in P_VALUES:
        t0 = time.time()
        key = (alpha_label, p)
        sol = solve_joint_p_median(demand_col, p=p, K=K0, lsoa_master=lsoa_master, coords=coords)
        xj = sol["xj"]
        eval_result = evaluate_allocation(demand_col, xj, sol["K_used"], lsoa_master, coords)
        core_results[key] = {**sol, **eval_result}
        core_summary_rows.append({
            "scenario": alpha_label, "p": p, "K": sol["K_used"],
            "status": sol["status"],
            "slack_total": sol["slack_total"],
            "slack_frac": sol["slack_frac"],
            "objective": eval_result["objective"],
            "n_lsoa_with_xj_gt_0": eval_result["n_lsoa_with_xj_gt_0"],
            "M1_avg_dist_m": eval_result["M1_avg_dist_m"],
            "M2_coverage_800m": eval_result["M2_coverage_800m"],
            "M3_imd_gap": eval_result["M3_imd_gap"],
            "M4_gini": eval_result["M4_gini"],
        })
        m1 = eval_result["M1_avg_dist_m"]; m2 = eval_result["M2_coverage_800m"]
        flag = "" if sol["slack_frac"] < SLACK_TOL else "  <-- slack exceeds SLACK_TOL"
        print(f"Scenario {alpha_label}, p={p}: status={sol['status']}, "
              f"slack={sol['slack_frac']:.2%}, M1={m1:.1f}m, M2={m2:.1%}, "
              f"time={time.time()-t0:.1f}s{flag}")

core_summary = pd.DataFrame(core_summary_rows)
print()
print("=== Round 3 grid: M1-M4 for all configurations ===")
print(core_summary.to_string(index=False))


## 5. Save results

In [ ]:
results_wide = lsoa_master[["lsoa_code", "ej"]].copy()
for (alpha_label, p), result in core_results.items():
    results_wide[f"xj_{alpha_label}_p{p}"] = result["xj"]
    results_wide[f"slack_{alpha_label}_p{p}"] = result["sj"]

output_path = os.path.join(BASE, "05_processed/p_median_results_round3.csv")
results_wide.to_csv(output_path, index=False)
print(f"Saved per-LSOA xj + slack: {output_path}")

run_status = core_summary[["scenario", "p", "K", "status", "slack_total", "slack_frac",
                           "objective", "M1_avg_dist_m", "M2_coverage_800m",
                           "M3_imd_gap", "M4_gini"]].copy()
run_status_path = os.path.join(BASE, "05_processed/p_median_run_status_round3.csv")
run_status.to_csv(run_status_path, index=False)
print(f"Saved per-config status/slack: {run_status_path}")
print(f"All configs Optimal: {(core_summary['status'] == 'Optimal').all()}, "
      f"max slack: {core_summary['slack_frac'].max():.2%}")


## 6. Diagnostics — equity mechanism check (primary evidence: r(xⱼ, income_score))

Kept as the headline equity evidence, per the Round 2 finding that this aggregate,
full-sample correlation behaved consistently (and correctly-signed) at every p tested so
far, unlike M3/per-decile M1 which were noisy at this scale.

In [ ]:
diag_table = results_wide.merge(imd_london[["lsoa_code", "income_score"]], on="lsoa_code", how="left")

print("=== Pearson r: xj (new capacity) vs income_score, by scenario, for every p tested ===")
for p in P_VALUES:
    print(f"\n-- p={p} --")
    for alpha_label in ["A", "B", "C", "D"]:
        col = f"xj_{alpha_label}_p{p}"
        r, p_val = pearsonr(diag_table[col], diag_table["income_score"])
        print(f"  Scenario {alpha_label}: r = {r:.4f}, p = {p_val:.4g}")


## 7. Secondary diagnostics — M3 and per-decile M1 (known to be noisy at scale, kept for completeness)

In [ ]:
pivot = core_summary.pivot(index="p", columns="scenario", values=["M1_avg_dist_m","M2_coverage_800m","M3_imd_gap","M4_gini"])
print("=== Full M1-M4 table, all configurations ===")
print(pivot.to_string())

print("\n=== Vi-weighted mean distance (m) by income decile, Scenario A vs Scenario D ===")
for p in P_VALUES:
    dist_A = core_results[("A", p)]["dist_by_decile"]
    dist_D = core_results[("D", p)]["dist_by_decile"]
    print(f"\n-- p={p} --")
    print(f"{'Decile':>6} {'A (eff.)':>10} {'D (equity)':>12} {'Delta':>10}")
    for dec in range(1, 11):
        a, d_ = dist_A[dec], dist_D[dec]
        delta = d_ - a
        tag = " <- most deprived" if dec == 1 else (" <- least deprived" if dec == 10 else "")
        print(f"{dec:>6} {a:>10.1f} {d_:>12.1f} {delta:>+10.1f}{tag}")


## 8. Xj map — "urgent area" view (raw new-capacity counts, Scenario C, p=PRIMARY_P)

Raw `xⱼ` values, not a ratio or per-capita rate — the point is exactly what was asked for:
LSOAs where `xⱼ = 0` need nothing further, LSOAs with high `xⱼ` are the urgent areas.
Scenario C (moderate equity, α=0.3) is used as the single map since it's the scenario the
proposal already treats as the primary one elsewhere; A vs D is kept below as a second,
comparative view.

In [ ]:
xj_col = f"xj_C_p{PRIMARY_P}"
map_data = lsoa_london.merge(results_wide[["lsoa_code", xj_col]], on="lsoa_code", how="left")

n_zero = (results_wide[xj_col] == 0).sum()
n_positive = (results_wide[xj_col] > 0).sum()
print(f"p={PRIMARY_P}, Scenario C: {n_zero:,} LSOAs need 0 new units, "
      f"{n_positive:,} LSOAs are allocated >0 (the 'urgent areas'), "
      f"max xj = {results_wide[xj_col].max()}")

fig, ax = plt.subplots(figsize=(11, 11))
map_data.plot(column=xj_col, cmap="OrRd", linewidth=0.1, edgecolor="grey",
              legend=True, ax=ax, legend_kwds={"label": f"New capacity units (xⱼ), p={PRIMARY_P}", "shrink": 0.6})
ax.set_title(f"Urgent Areas: New EV Charging Capacity Allocation (Scenario C, p={PRIMARY_P})", fontsize=13)
ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(BASE, f"06_outputs/figures/fig_xj_urgent_areas_C_p{PRIMARY_P}.png"), dpi=300, bbox_inches="tight")
plt.show()


## 8b. Allocation maps — Scenario A vs Scenario D (p=PRIMARY_P), for comparison

In [ ]:
map_data_ad = lsoa_london.merge(
    results_wide[["lsoa_code", f"xj_A_p{PRIMARY_P}", f"xj_D_p{PRIMARY_P}"]], on="lsoa_code", how="left"
)
solA = core_results[("A", PRIMARY_P)]; solD = core_results[("D", PRIMARY_P)]

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
map_data_ad.plot(column=f"xj_A_p{PRIMARY_P}", cmap="OrRd", linewidth=0.1, edgecolor="grey",
              legend=True, ax=axes[0], legend_kwds={"label": "New capacity units (xj)", "shrink": 0.6})
axes[0].set_title(f"Scenario A (alpha=0) — Efficiency Baseline, p={PRIMARY_P}\n"
                  f"status={solA['status']}, slack={solA['slack_frac']:.2%}"); axes[0].axis("off")

map_data_ad.plot(column=f"xj_D_p{PRIMARY_P}", cmap="OrRd", linewidth=0.1, edgecolor="grey",
              legend=True, ax=axes[1], legend_kwds={"label": "New capacity units (xj)", "shrink": 0.6})
axes[1].set_title(f"Scenario D (alpha=0.5) — Strong Equity, p={PRIMARY_P}\n"
                  f"status={solD['status']}, slack={solD['slack_frac']:.2%}"); axes[1].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(BASE, "06_outputs/figures/fig_allocation_A_vs_D_round3.png"), dpi=300, bbox_inches="tight")
plt.show()


## 9. Update pipeline_summary.csv

In [ ]:
pipeline_summary = pd.read_csv(os.path.join(BASE, "05_processed/pipeline_summary.csv"))
item = "P-median allocation (joint MILP, Round 3 — real-world target range)"
value = f"Done — {len(core_results)} configs, K0={K0:.4f}, P_VALUES={P_VALUES}, max slack={core_summary['slack_frac'].max():.2%}"
if (pipeline_summary["Item"] == item).any():
    pipeline_summary.loc[pipeline_summary["Item"] == item, "Count"] = value
else:
    pipeline_summary = pd.concat([pipeline_summary, pd.DataFrame([{"Item": item, "Count": value}])], ignore_index=True)
pipeline_summary.to_csv(os.path.join(BASE, "05_processed/pipeline_summary.csv"), index=False)
print(pipeline_summary.to_string(index=False))
